# Modeling Nonlinear Retail Demand Curves with PROC GAM

## Executive Summary

This notebook uses **PROC GAM** to model weekly grocery unit sales as a smooth, nonlinear function of shelf price, store temperature (a seasonality proxy), and promotional spend, with a parametric promotion-flag effect. Generalized additive models let a category manager recover the true curved price-elasticity and seasonal-demand shapes that a linear regression would miss, supporting sharper pricing and promotion decisions.

## Data Sources

| Dataset | Rows | Grain | Key Variables | Description |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | week | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Synthetic weekly point-of-sale series for one flagship grocery store across 100 consecutive weeks (nearly two seasonal cycles). Generated inline with `call streaminit(20250531)` and `rand()`. Weekly unit sales follow a Poisson demand process whose mean is driven by an exponential price-response curve, a quadratic temperature/seasonality effect peaking near 72F, and a concave (square-root) promotion-spend lift, plus a discrete promotion-week flag. |

# Modeling Nonlinear Retail Demand Curves with PROC GAM

Retail demand rarely responds to price, weather, or promotion spend in a straight line. A small price cut on a staple may barely move volume, while crossing a psychological price point can trigger a sharp jump; weather-driven demand peaks in a comfortable mid-range and falls off at both extremes; and promotional lift shows diminishing returns as spend climbs.

**PROC GAM** (generalized additive models) fits each driver with its own smooth spline term, so the data — not a rigid linear assumption — determines the shape of every demand curve. Here we model weekly unit sales for a single flagship grocery store across 100 consecutive weeks, combining a parametric promotion flag with smoothing splines on price, temperature, and promotion spend under a Poisson response.

## Step 1 — Generate a synthetic weekly sales series

We simulate 100 consecutive weeks (nearly two seasonal cycles) of point-of-sale data for one flagship store. The data-generating process is deliberately nonlinear so we can confirm that GAM recovers realistic shapes:

- **Price** drives volume through an exponential response curve (`exp(-1.1 * Price)`), so demand rises steeply as price falls.
- **Temperature** acts as a seasonality proxy with a quadratic peak near 72F, modeling comfort-driven foot traffic.
- **PromoSpend** delivers a concave, square-root lift (diminishing returns).
- A discrete **Promotion** flag adds a parametric step effect on promoted weeks.

Weekly `Units` are drawn from a Poisson distribution, matching the count nature of unit sales.

In [1]:
data store_sales;
   call streaminit(20250531);
   length Promotion $3;
   do Week = 1 to 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      if rand("uniform") < 0.28 then do;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      end;
      else do;
         Promotion  = "No";
         PromoSpend = 0;
      end;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * exp(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + log(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", exp(logMu) / 12);
      output;
   end;
run;

NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Step 2 — Profile the simulated data

A quick `PROC MEANS` confirms the variables span sensible retail ranges before modeling: unit counts are non-negative integers, price clusters around a few dollars, temperature sweeps a full seasonal cycle, and promotion spend is zero on non-promoted weeks.

In [2]:
proc means data=store_sales n mean min max maxdec=2;
   var Units Price Temperature PromoSpend;
run;

                                                  The MEANS Procedure

 Variable            N           Mean     Minimum     Maximum
 ------------------------------------------------------------
 Units             100           7.03        0.00      103.00
 Price             100           3.15        2.81        3.48
 Temperature       100          55.50       22.72       83.49
 PromoSpend        100         128.76        0.00      779.00
 ------------------------------------------------------------



NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Step 3 — Fit the full additive demand model

The core model combines:

- `param(Promotion)` — a parametric (linear) effect for the promotion-week indicator, declared on the `CLASS` statement.
- `spline(Price, df=4)` — a cubic smoothing spline capturing the curved price-response.
- `spline(Temperature, df=4)` — a smooth seasonality curve.
- `spline(PromoSpend, df=3)` — diminishing-returns promotion lift.

Because unit sales are counts, we specify `dist=poisson` (log link). `method=gcv` lets generalized cross-validation guide the smoothness of any component without an explicit degrees-of-freedom override. The `OUTPUT` statement writes per-observation predictions and residuals to `gam_fit`.

The procedure reports a **Deviance of 43.59** against a **Null Deviance of 2041.12** — the four smooth-plus-parametric drivers explain almost all of the variation a constant-only model leaves on the table — and an **AIC of 254.61**. The parametric `PROMOTIONYES` estimate is positive (+0.41 on the log scale), confirming the promotional volume lift, and the price spline carries a strongly negative linear trend (−1.71), the signature of downward-sloping demand.

In [3]:
proc gam data=store_sales plots=components(clm commonaxes);
   class Promotion;
   model Units = param(Promotion)
                 spline(Price, df=4)
                 spline(Temperature, df=4)
                 spline(PromoSpend, df=3) / dist=poisson method=gcv;
   output out=gam_fit predicted residual;
run;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     UNITS
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Price)                    4.0000         4.

NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Step 4 — A focused price-and-seasonality model

For a leaner pricing review, we refit with only the two most decision-relevant smooth drivers — price and temperature — giving price extra flexibility (`df=5`) to resolve any kink near a psychological price point. The promotion flag is retained as a parametric control.

Dropping promotion spend raises the **Deviance to 62.80** and the **AIC to 269.82**, both higher than the full model's 43.59 and 254.61. The parametric `PROMOTIONYES` term also absorbs more of the promotional signal here (+1.73 versus +0.41), since the spend spline is no longer present to carry it. The price spline keeps its negative trend (−1.51), so the core demand story is stable across specifications.

In [4]:
proc gam data=store_sales plots=components(clm);
   class Promotion;
   model Units = param(Promotion)
                 spline(Price, df=5)
                 spline(Temperature, df=4) / dist=poisson;
run;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     UNITS
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Price)                    5.0000         5.0000
Spline(Temperature)              4.0000         4.0000




NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Interpreting the results

The **Regression Model Analysis** table reports the linear trend within each component plus the parametric promotion effect. The positive `PROMOTIONYES` coefficient (+0.41 in the full model, +1.73 in the leaner one) confirms the expected promotional volume lift, while the negative linear trend on the price spline (−1.71 and −1.51) reflects classic downward-sloping demand. The temperature spline's small positive linear term (+0.03) is expected: its real story is curvature around the 72F comfort peak, which a single linear coefficient cannot summarize.

The **Smoothing Model Analysis** table reports the degrees of freedom each spline consumes. Price and temperature each draw 4 effective DF (5 for price in the leaner model) and promotion spend 3 — each well above the single DF a straight line would use, which is exactly why a linear regression would miss these curved relationships.

The **Fit Statistics** let you compare the two specifications directly. The full four-driver model posts a Deviance of 43.59 and AIC of 254.61 against the leaner model's 62.80 and 269.82; both criteria favor the full model, showing that promotion spend and the promotion flag add explanatory power beyond price and temperature alone. Relative to the Null Deviance of 2041.12, both models capture the overwhelming majority of demand variation.

Read together, these tables give a category manager a quantified, data-driven demand story: a steep price response that informs markdown depth, a seasonal temperature window, and a diminishing-returns promotion-spend effect — far sharper guidance than a single linear elasticity estimate. (PROC GAM also accepts `plots=components` to render the partial-prediction curves for each smooth term; the numeric component tables above are the source those curves are drawn from.)